# 3. Call the deployed model

Sends fresh simulated windows for each condition to the model's KServe v2 (Open Inference Protocol) REST endpoint.
Set `MODEL_URL` to the inference endpoint shown in the dashboard (internal or external). If unset, the notebook scores locally with ONNX Runtime so you can still test the flow.

In [ ]:
# Install the demo package from the cloned repo (skipped if already installed)
import importlib.util
if importlib.util.find_spec("bearing") is None:
    %pip install -q -e "..[notebook]"
    print("Installed. If imports fail, restart the kernel once.")

In [ ]:
import os, time
import numpy as np
from bearing.simulate import simulate_window
from bearing.features import extract_features
from bearing.inference import KServeV2Client, LocalOnnxClient, label

MODEL_URL = os.environ.get("MODEL_URL", "")   # e.g. https://bearing-<project>.apps.<cluster>
if MODEL_URL:
    client = KServeV2Client(MODEL_URL, token=os.environ.get("MODEL_TOKEN"),
                            verify_tls=os.environ.get("MODEL_VERIFY_TLS", "true") != "false")
    print("calling", client.url)
else:
    client = LocalOnnxClient("../models/bearing/1/model.onnx")
    print("MODEL_URL not set: scoring locally with ONNX Runtime")

In [ ]:
rng = np.random.default_rng(42)
for mode, sev in [("healthy", 0), ("imbalance", 0.3), ("imbalance", 0.9), ("bearing_wear", 0.2), ("bearing_wear", 0.9)]:
    w = simulate_window(mode, sev, rng)
    x = extract_features(w["samples"], w["temp_c"], w["current_a"])
    t0 = time.perf_counter(); probs = client.predict_proba(x); ms = (time.perf_counter() - t0) * 1000
    (pred, conf), = label(probs)
    print(f"truth={mode:<13} severity={sev:.1f}  ->  {pred:<13} {conf:.2f}   ({ms:.1f} ms)")

## Raw request, for curl or Postman
```
POST {MODEL_URL}/v2/models/bearing/infer
{"inputs": [{"name": "features", "shape": [1, 10], "datatype": "FP32", "data": [ ...10 numbers... ]}]}
```

In [ ]:
import json
w = simulate_window("bearing_wear", 0.8, rng)
x = extract_features(w["samples"], w["temp_c"], w["current_a"])
print(json.dumps({"inputs": [{"name": "features", "shape": [1, 10], "datatype": "FP32", "data": [round(float(v), 4) for v in x]}]}))